In [2]:
from trino.dbapi import connect
import pandas as pd
from minio import  Minio
import io, time, warnings
from io import BytesIO
warnings.filterwarnings('ignore')
# buffer = io.BytesIO()

In [3]:
bucket_name = "padakhep-lakehouse"
client = Minio(
    '202.4.114.56:9000',
    access_key='gazu',
    secret_key='asdf1234',
    secure=False,
)

conn = connect(
    host = "202.4.114.56",
    port = "8888",
    user = "gazu",
    catalog= "padakhep",
    schema= "gold",
)
cur = conn.cursor()

def query_to_df(sql):
    cur.execute(sql)
    cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

In [4]:
#updated one
def upload_to_minio(data, object_name):
    buffer = BytesIO()

    # Convert DataFrame to Parquet
    data.to_parquet(
        buffer,
        index=False,
        engine="pyarrow"
    )

    # Important: go back to the beginning
    buffer.seek(0)

    file_size = buffer.getbuffer().nbytes

    # Upload to MinIO
    client.put_object(
        bucket_name=bucket_name,
        object_name=object_name,
        data=buffer,
        length=file_size,
        content_type="application/vnd.apache.parquet",
    )

    print(f"Uploaded: s3://{bucket_name}/{object_name} ({file_size:,} bytes)")

    buffer.close()


In [5]:
# member_id = 4686
# start_date = '2026-01-01'
# end_date = '2026-01-30'
#
# # Step 1: find distinct ls.id values for this member/date range
# q1 =(f"""
#     SELECT DISTINCT ls.id
#     FROM loans
#     LEFT JOIN loan_schedules ls ON ls.loan_id = loans.id
#     WHERE loans.member_id = {member_id}
#       AND loans.inactive_date >= DATE '{start_date}'
#       AND loans.inactive_date < DATE '{end_date}'
# """)
# =query_to_df(q1)
#
# cur.execute(f"""
#     SELECT DISTINCT ls.id
#     FROM loans
#     LEFT JOIN loan_schedules ls ON ls.loan_id = loans.id
#     WHERE loans.member_id = {member_id}
#       AND loans.inactive_date >= DATE '{start_date}'
#       AND loans.inactive_date < DATE '{end_date}'
# """)

In [6]:
# schedule_ids = [row[0] for row in cur.fetchall() if row[0] is not None]
#
# if not schedule_ids:
#     print("No schedules found for this member/date range.")
# else:
#     # Step 2: build the pivot columns dynamically in Python
#     case_columns = ",\n  ".join(
#         f'MAX(CASE WHEN ls.id = {sid} THEN ls.id END) AS "schedule_{sid}"'
#         for sid in schedule_ids
#     )

In [12]:
# final_query = f"""
#     SELECT loans.member_id,
#       {case_columns}
#     FROM loans
#     LEFT JOIN loan_schedules ls ON ls.loan_id = loans.id
#     WHERE loans.member_id = {member_id}
#       AND loans.inactive_date >= DATE '{start_date}'
#       AND loans.inactive_date < DATE '{end_date}'
#     GROUP BY loans.member_id
# """
#
# # Step 3: run it as a plain SELECT
# cur.execute(final_query)
# columns = [desc[0] for desc in cur.description]
# result = cur.fetchall()
#
# print(columns)
# for row in result:
#     print(row)

In [13]:
# def query_to_df(sql):
#     cur.execute(sql)
#     cols = [desc[0] for desc in cur.description]
#     return pd.DataFrame(cur.fetchall(), columns=cols)
#
#
# member_id = 4686
# start_date = '2026-01-01'
# end_date = '2026-01-30'
#
# # Step 1: rank each schedule per member
# ranked_sql = f"""
#     SELECT
#       loans.member_id,
#       ls.id AS schedule_id,
#       ROW_NUMBER() OVER (PARTITION BY loans.member_id ORDER BY ls.id) AS payment_num
#     FROM loans
#     LEFT JOIN loan_schedules ls ON ls.loan_id = loans.id
#     WHERE loans.member_id = {member_id}
#       AND loans.inactive_date >= DATE '{start_date}'
#       AND loans.inactive_date < DATE '{end_date}'
# """
#
# ranked_df = query_to_df(ranked_sql)
#
# if ranked_df.empty or ranked_df['payment_num'].isna().all():
#     print("No matching schedules found for this member/date range.")
#     result_df = pd.DataFrame()  # empty, so nothing downstream breaks
# else:
#     max_payments = int(ranked_df['payment_num'].max())
#
#     case_columns = ",\n  ".join(
#         f'MAX(CASE WHEN rn.payment_num = {n} THEN rn.schedule_id END) AS "payment_{n}"'
#         for n in range(1, max_payments + 1)
#     )
#
#     final_sql = f"""
#         WITH ranked AS (
#           SELECT
#             loans.member_id,
#             ls.id AS schedule_id,
#             ROW_NUMBER() OVER (PARTITION BY loans.member_id ORDER BY ls.id) AS payment_num
#           FROM loans
#           LEFT JOIN loan_schedules ls ON ls.loan_id = loans.id
#           WHERE loans.member_id = {member_id}
#             AND loans.inactive_date >= DATE '{start_date}'
#             AND loans.inactive_date < DATE '{end_date}'
#         )
#         SELECT
#           rn.member_id,
#           {case_columns}
#         FROM ranked rn
#         GROUP BY rn.member_id
#     """
#
#     result_df = query_to_df(final_sql)
#
# result_df

In [14]:
# def query_to_df(sql):
#     cur.execute(sql)
#     cols = [desc[0] for desc in cur.description]
#     return pd.DataFrame(cur.fetchall(), columns=cols)
#
#
# member_ids = [1168348, 6318, 7029]   # <-- your list of member ids
# start_date = '2025-01-01'
# end_date = '2026-06-30'
#
# member_ids_str = ", ".join(str(m) for m in member_ids)
#
# # Step 1: rank each schedule per member
# ranked_sql = f"""
#     SELECT
#       loans.member_id,
#       ls.id AS schedule_id,
#       ROW_NUMBER() OVER (PARTITION BY loans.member_id ORDER BY ls.id) AS payment_num
#     FROM loans
#     LEFT JOIN loan_schedules ls ON ls.loan_id = loans.id
#     WHERE loans.member_id IN ({member_ids_str})
#       AND loans.inactive_date >= DATE '{start_date}'
#       AND loans.inactive_date < DATE '{end_date}'
#       AND ls.is_holiday = 'False'
# """
#
# ranked_df = query_to_df(ranked_sql)
#
# if ranked_df.empty or ranked_df['payment_num'].isna().all():
#     print("No matching schedules found for these members/date range.")
#     result_df = pd.DataFrame()
# else:
#     max_payments = int(ranked_df['payment_num'].max())
#
#     case_columns = ",\n  ".join(
#         f'MAX(CASE WHEN rn.payment_num = {n} THEN rn.schedule_id END) AS "payment_{n}"'
#         for n in range(1, max_payments + 1)
#     )
#
#     final_sql = f"""
#         WITH ranked AS (
#           SELECT
#             loans.member_id,
#             ls.id AS schedule_id,
#             ROW_NUMBER() OVER (PARTITION BY loans.member_id ORDER BY ls.id) AS payment_num
#           FROM loans
#           LEFT JOIN loan_schedules ls ON ls.loan_id = loans.id
#           WHERE loans.member_id IN ({member_ids_str})
#             AND loans.inactive_date >= DATE '{start_date}'
#             AND loans.inactive_date < DATE '{end_date}'
#         )
#         SELECT
#           rn.member_id,
#           {case_columns}
#         FROM ranked rn
#         GROUP BY rn.member_id
#         ORDER BY rn.member_id
#     """
#
#     result_df = query_to_df(final_sql)
#
# result_df

### Final

In [18]:
# def query_to_df(sql):
#     cur.execute(sql)
#     cols = [desc[0] for desc in cur.description]
#     return pd.DataFrame(cur.fetchall(), columns=cols)
#
#
# # member_ids = [4686, 6318, 7029]
# start_date = '2025-01-01'
# end_date = '2026-06-30'
# disbursement_date = '2025-01-01'
# # member_ids_str = ", ".join(str(m) for m in member_ids)
#
# ranked_sql = f"""
#     SELECT
#       loans.member_id,
#       ls.id AS schedule_id,
#       ROW_NUMBER() OVER (PARTITION BY loans.member_id ORDER BY ls.id) AS payment_num
#     FROM loans
#     LEFT JOIN loan_schedules ls ON ls.loan_id = loans.id
#     WHERE
#       --loans.member_id IN (member_ids_str)
#        loans.inactive_date >= DATE '{start_date}'
#       AND loans.inactive_date < DATE '{end_date}'
#       and is_holiday ='False'
# """
#
# ranked_df = query_to_df(ranked_sql)
#
# if ranked_df.empty or ranked_df['payment_num'].isna().all():
#     print("No matching schedules found.")
#     result_df = pd.DataFrame()
# else:
#     max_payments = int(ranked_df['payment_num'].max())
#
#     case_columns = ",\n      ".join(
#         f'MAX(CASE WHEN rn.payment_num = {n} THEN rn.paid_pattern END) AS "payment_{n}"'
#         for n in range(1, max_payments + 1)
#     )
#
#     final_sql = f"""
#         WITH ranked AS (
#           SELECT
#             m.name,
#             m.gender,
#             m.dob DateOFBirth,
#             m.admission_date Admission_Date,
#             m.occupation MemberOccupation,
#             m.marital_status Maritial_Status,
#             loans.id  as loan_id,
#             loans.disbursed_date,
#             loans.loan_amount AS loan_amount,
#             MIN(ls.due_date) OVER (PARTITION BY loans.member_id) AS start_date,
#             MAX(ls.due_date) OVER (PARTITION BY loans.member_id) AS end_date,
#             MAX(ls.kisti_paid_date) OVER (PARTITION BY loans.member_id) AS Last_paid_date,
#             loans.member_id,
#             ls.id AS schedule_id,
#             loan_products.inst_type AS InstallmentType,
#             loan_products.inst_total AS TotalInstallmentCount,
#             loan_applications.dofa_no AS Loan_Cycle_No,
#             loan_components.name AS Loan_Component,
#             CASE WHEN due_date = kisti_paid_date and kisti_amount = paid_amount then 'Regular'
# 		WHEN due_date > kisti_paid_date and kisti_amount = paid_amount then 'Advance'
# 		ELSE 'Bokeya' END paid_pattern,
#            --  ls.kisti_amount AS kisti_amount,
#             ROW_NUMBER() OVER (PARTITION BY loans.member_id ORDER BY ls.id) AS payment_num
#           FROM loans
#           LEFT JOIN loan_schedules ls ON ls.loan_id = loans.id
#           LEFT JOIN members m ON m.id = loans.member_id
#           LEFT JOIN loan_products ON loan_products.id = loans.loan_product_id
#           LEFT JOIN loan_applications ON loan_applications.id = loans.loan_application_id
#           LEFT JOIN loan_components ON loan_components.id = loan_products.component_id
#           WHERE
#              -- loans.member_id IN (member_ids_str)
#              loans.inactive_date >= DATE '{start_date}'
#             AND loans.inactive_date < DATE '{end_date}'
#         )
#         SELECT
#           rn.member_id,
#           rn.name,
#           rn.gender,
#           rn.Maritial_Status,
#           rn.MemberOccupation,
#           rn.DateOFBirth,
#           rn.Admission_Date,
#           rn.Loan_Cycle_No,
#           rn.loan_id,
#           rn.Loan_Component,
#           rn.InstallmentType,
#           rn.TotalInstallmentCount,
#           rn.loan_amount,
#           rn.disbursed_date,
#           rn.start_date,
#           rn.end_date,
#           rn.Last_paid_date,
#           {case_columns}
#         FROM ranked rn
#         GROUP BY rn.member_id, rn.name, rn.loan_amount, rn.start_date, rn.end_date, rn.loan_id, rn.disbursed_date, rn.Last_paid_date,rn.gender,
#          rn.Maritial_Status,
#           rn.MemberOccupation,
#           rn.DateOFBirth,
#           rn.Admission_Date,
#           rn.Loan_Cycle_No,
#           rn.Loan_Component,
#           rn.InstallmentType,
#           rn.TotalInstallmentCount
#         ORDER BY rn.member_id
#     """
#
#     # print(final_sql)   # <-- inspect this if it errors again
#     result_df = query_to_df(final_sql)
#
# result_df

In [18]:
import time

def query_to_df(sql):
    cur.execute(sql)
    cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)


# start_date = '2025-01-01'
end_date = '2026-07-01'
# -disbursement_date = '2024-06-01'
loan_limit = 300000
batch_size = 20000   # tune this based on how it performs

# Step 1: get max_payments once (small, cheap query, scoped to the same loan set)
ranked_sql = f"""
    WITH selected_loans AS (
      SELECT loans.id AS loan_id
      FROM loans
      WHERE
      -- loans.inactive_date >= DATE 'start_date'
        loans.inactive_date < DATE '{end_date}'
        AND loans.disbursed_date < DATE '{end_date} '
        and loans.id = 1861037
        AND  (
        loans.recovery_method = 'flat'
        OR loans.recovery_method IS NULL
         )
      ORDER BY loans.disbursed_date DESC , loans.id DESC -- add later to remove duplicate
      LIMIT {loan_limit}
    )
    SELECT
      loans.member_id,
      ls.id AS schedule_id,
      ROW_NUMBER() OVER (PARTITION BY loans.id ORDER BY ls.id) AS payment_num
    FROM loans
    JOIN selected_loans sl ON sl.loan_id = loans.id
    LEFT JOIN loan_schedules ls ON ls.loan_id = loans.id
    WHERE ls.is_holiday = 'False'
"""
ranked_df = query_to_df(ranked_sql)
max_payments = int(ranked_df['payment_num'].max())

case_columns = ",\n      ".join(
    f'MAX(CASE WHEN rn.payment_num = {n} THEN rn.paid_pattern END) AS "installment_{n}"'
    for n in range(1, max_payments + 1)
)

# Step 2: pull loans in batches, using OFFSET/LIMIT on the same ordered selected_loans set
all_batches = []
offset = 0

while offset < loan_limit:
    this_batch_size = min(batch_size, loan_limit - offset)

    final_sql = f"""
        WITH selected_loans AS (
          SELECT loans.id AS loan_id
          FROM loans
          WHERE
         loans.id = 1861037
         and  (
            loans.recovery_method = 'flat'
            OR loans.recovery_method IS NULL
        )  and
            -- loans.inactive_date >= DATE 'start_date' AND
            loans.inactive_date < DATE '{end_date}'
            AND loans.disbursed_date < DATE '{end_date}'
          ORDER BY loans.disbursed_date DESC, loans.id DESC   -- add later to remove duplicate
          OFFSET {offset} LIMIT {this_batch_size}
        ),
        ranked AS (
          SELECT
            m.name, m.gender, m.dob AS dateofbirth,
            m.admission_date AS admission_date,
            m.occupation AS occupation,
            m.occupation_alt alternative_occupation,
            m.marital_status AS maritial_status,
            m.education AS education,
            m.total_family_member,
            m.total_earning_member,
            m.total_monthly_income,
            m.yearly_total_income,
            m.yearly_total_expenses,
            m.yearly_total_savings,
            m.home_owner_type,
            m.year_of_rent,
            m.farm_land_amt,
            m.loan_use_sources,
            -- new info added
            -- LAG (loans.loan_amount) over (partition by m.id order by dofa_no ) as prev_loan_amount,

            loans.id AS loan_id,
            loans.disbursed_date,
            loans.loan_amount AS loan_amount,
            MIN(ls.due_date) OVER (PARTITION BY loans.id) AS start_date,
            MAX(ls.due_date) OVER (PARTITION BY loans.id) AS end_date,
            MAX(ls.kisti_paid_date) OVER (PARTITION BY loans.id) AS last_paid_date,
            loans.member_id,
            ls.id AS schedule_id,
            loan_products.inst_type AS installmenttype,
            loan_products.inst_total AS totalInstallmentCount,
            loan_applications.dofa_no AS loan_cycle_no,
            loan_components.name AS loan_component,
            -- COALESCE(count(tl.id), 0) transaction_count,
            CASE
              WHEN due_date = kisti_paid_date AND kisti_amount = paid_amount THEN 'Regular'
              WHEN due_date > kisti_paid_date AND kisti_amount = paid_amount THEN 'Advance'
              ELSE 'Bokeya'
            END AS paid_pattern,
            ROW_NUMBER() OVER (PARTITION BY loans.id ORDER BY ls.id) AS payment_num
          FROM loans
          JOIN selected_loans sl ON sl.loan_id = loans.id
          LEFT JOIN loan_schedules ls ON ls.loan_id = loans.id and ls.is_holiday = 'False'
          LEFT JOIN members m ON m.id = loans.member_id
          LEFT JOIN loan_products ON loan_products.id = loans.loan_product_id
          LEFT JOIN loan_applications ON loan_applications.id = loans.loan_application_id
          LEFT JOIN loan_components ON loan_components.id = loan_products.component_id
        --   LEFT JOIN transactions_loans tl ON tl.loan_schedule_id = ls.id
        WHERE ls.is_holiday = 'False'
        ),
        txn_counts AS (
      SELECT
        ls.loan_id,
        COUNT(tl.id) AS transaction_count,
        count(distinct date_of_action) paymentdaycount
      FROM loan_schedules ls
      JOIN selected_loans sl ON sl.loan_id = ls.loan_id
      LEFT JOIN transactions_loans tl ON tl.loan_schedule_id = ls.id
       where ls.is_holiday = 'False'
      GROUP BY ls.loan_id
    ),

        loan_level AS (
      SELECT DISTINCT
        l.id AS loan_id,
        l.member_id,
        l.loan_amount,
        loan_applications.dofa_no
      FROM loans l
      LEFT JOIN loan_applications ON loan_applications.id = l.loan_application_id
        WHERE l.member_id IN (
    SELECT DISTINCT l2.member_id
    FROM loans l2
    JOIN selected_loans sl ON sl.loan_id = l2.id
  )
    ),

    loan_level_lag AS (
  SELECT
    loan_id,
        LAG(loan_amount) OVER (PARTITION BY member_id ORDER BY dofa_no) AS prev_loan_amount,
        LAG(dofa_no) OVER (PARTITION BY member_id ORDER BY dofa_no) AS prev_cycle_no
      FROM loan_level
    )

        SELECT
    -- Member Information
    rn.member_id,
    rn.name,

    -- Personal Information
    rn.dateofbirth,
    rn.gender,
    rn.maritial_status,
    rn.admission_date,

    -- Education & Occupation
    rn.education,
    rn.occupation,
    rn.alternative_occupation,

    -- Family & Financial Information
    rn.total_family_member,
    rn.total_earning_member,
    rn.total_monthly_income,
    rn.yearly_total_income,
    rn.yearly_total_expenses,
    rn.yearly_total_savings,

    -- Housing & Assets
    rn.home_owner_type,
    rn.year_of_rent,
    rn.farm_land_amt,

    -- Loan Information
    rn.loan_id,
    rn.loan_cycle_no,
    ll.prev_cycle_no,
    rn.loan_Component,
    rn.loan_use_sources,
    rn.loan_amount,
    ll.prev_loan_amount,

    -- Loan Schedule & Repayment
    rn.disbursed_date,
    rn.start_date,
    rn.end_date,
    rn.installmenttype,
    rn.totalInstallmentcount,
    tc.transaction_count,
    tc.paymentdaycount,
    rn.last_paid_date,

    -- Calculated / Case Information
    {case_columns}

FROM ranked rn
 LEFT JOIN txn_counts tc ON tc.loan_id = rn.loan_id
LEFT JOIN loan_level_lag ll ON ll.loan_id = rn.loan_id

GROUP BY
    rn.member_id,
    rn.name,
    rn.dateofbirth,
    rn.gender,
    rn.maritial_status,
    rn.admission_date,
    rn.education,
    rn.occupation,
    rn.alternative_occupation,
    rn.total_family_member,
    rn.total_earning_member,
    rn.total_monthly_income,
    rn.yearly_total_income,
    rn.yearly_total_expenses,
    rn.yearly_total_savings,
    rn.home_owner_type,
    rn.year_of_rent,
    rn.farm_land_amt,
    rn.loan_id,
    rn.loan_cycle_No,
    rn.loan_component,
    rn.loan_use_sources,
    rn.loan_amount,
    rn.disbursed_date,
    rn.start_date,
    rn.end_date,
    rn.installmenttype,
    rn.totalInstallmentcount,
    rn.last_paid_date,
    ll.prev_loan_amount,
    ll.prev_cycle_no,
    tc.transaction_count,
    tc.paymentdaycount


ORDER BY
    rn.disbursed_date DESC
    """

    try:
        batch_df = query_to_df(final_sql)
    except Exception as e:
        print(f"Batch failed at offset {offset}: {e}")
        break

    if batch_df.empty:
        break
    batch_df.head()
    all_batches.append(batch_df)
    print(f"Fetched offset {offset}–{offset + len(batch_df)} ({len(batch_df)} rows)")

    offset += this_batch_size
    time.sleep(1)   # small pause between batches, easy on the server

result_df = pd.concat(all_batches, ignore_index=True) if all_batches else pd.DataFrame()
result_df

Fetched offset 0–1 (1 rows)


,member_id,name,dateofbirth,gender,maritial_status,admission_date,education,occupation,alternative_occupation,total_family_member,...,installment_37,installment_38,installment_39,installment_40,installment_41,installment_42,installment_43,installment_44,installment_45,installment_46
0,948498,Hasina begum,1979-06-15,Female,Married,2024-05-19,Primary,Housewife,Housewife,4,...,Regular,Advance,Advance,Advance,Advance,Advance,Advance,Advance,Advance,Advance


In [16]:
object_name= "datamart/Credit_Worthiness/Schedule_Pattern_Dataset.parquet"
upload_to_minio(result_df,object_name)

Uploaded: s3://padakhep-lakehouse/datamart/Credit_Worthiness/Schedule_Pattern_Dataset.parquet (30,833,654 bytes)


In [17]:
result_df.to_csv("schedule_Pattern_Dataset.csv", index=False, encoding="utf-8-sig")

In [18]:
pd.read_csv('schedule_Pattern_Dataset.csv')

,member_id,name,dateofbirth,gender,maritial_status,admission_date,education,occupation,alternative_occupation,total_family_member,...,installment_82,installment_83,installment_84,installment_85,installment_86,installment_87,installment_88,installment_89,installment_90,installment_91
0,302095,Mst:Nasima Begum,1985-08-19,Female,Married,2019-05-06,Pre-Primary,Agriculture,krisi,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1339749,NAJMA BEGUM,1984-01-01,Female,Married,2026-06-03,Secondary,Housewife,NaN,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1137258,Mohammad Mofiz,1968-05-10,Male,Married,2025-05-07,Higher-Secondary,Business,Business,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1186424,Md. Kamal Hossain Mrida,1969-03-05,Male,Married,2025-08-03,Secondary,Business,NaN,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1336203,SABINA KHATUN,1996-08-04,Female,Married,2026-05-17,Pre-Primary,Housewife,NaN,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299865,1029977,NUR NAHAR,1992-01-01,Female,Married,2024-11-11,Pre-Primary,Housewife,NaN,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
299866,1029012,Manjura khatun,1998-07-15,Female,Married,2024-11-10,Pre-Primary,Housewife,NaN,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
299867,1014458,Shahera Begum,1989-07-10,Female,Married,2024-10-20,Primary,Housewife,NaN,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
299868,450434,Mst. Shathy Begum,1989-01-08,Female,Married,2021-12-08,Higher-Secondary,Business,NaN,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# import xlsxwriter
# workbook = xlsxwriter.Workbook(
#     "../August-2026/CM_Loan_Informations_July(amortization).xlsx"
# )
# workbook.use_zip64()
# result_df.write_excel(workbook=workbook)
# workbook.close()

In [5]:
query_to_df(f"""
    SELECT ls.loan_id, COUNT(*) AS row_count
    FROM loan_schedules ls
    LEFT JOIN transactions_loans tl ON tl.loan_schedule_id = ls.id
    WHERE ls.loan_id = 1738943
    GROUP BY ls.loan_id
""")

,loan_id,row_count
0,1738943,14


In [10]:
query_to_df("SELECT id, COUNT(*) FROM members GROUP BY id HAVING COUNT(*) > 1")
query_to_df("SELECT id, COUNT(*) FROM loan_products GROUP BY id HAVING COUNT(*) > 1")
query_to_df("SELECT id, COUNT(*) FROM loan_applications GROUP BY id HAVING COUNT(*) > 1")
query_to_df("SELECT id, COUNT(*) FROM loan_components GROUP BY id HAVING COUNT(*) > 1")

,id,_col1


In [11]:
query_to_df(f"""
    SELECT
      ls.loan_id,
      COUNT(tl.id) AS transaction_count,
      COUNT(DISTINCT date_of_action) AS paymentdaycount
    FROM loan_schedules ls
    LEFT JOIN transactions_loans tl ON tl.loan_schedule_id = ls.id
    WHERE ls.loan_id = 1738943
      AND ls.is_holiday = 'False'
    GROUP BY ls.loan_id
""")

,loan_id,transaction_count,paymentdaycount
0,1738943,13,11
